<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Ai/blob/main/CatVSDog_using_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐶 Cat vs Dog Classification using CNN in PyTorch

A binary image classification project built from scratch using PyTorch.

Dataset: Microsoft Cats & Dogs Dataset

Model: Custom CNN

Framework: PyTorch

## Objective

Build a Convolutional Neural Network that classifies whether an image contains a cat or a dog.

This notebook demonstrates:

- Image preprocessing
- Dataset loading
- CNN architecture
- Model training
- Evaluation

## Dataset

The dataset contains images of cats and dogs.

Classes:
- Cat
- Dog

Images are resized to 224×224 before training.

In [25]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'dog-and-cat-classification-dataset' dataset.
Path to dataset files: /kaggle/input/dog-and-cat-classification-dataset


In [26]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset , DataLoader

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [28]:
from torchvision.datasets import ImageFolder
from torchvision import transforms

## Image Transformations

Images are resized to 224 only in 1 Dim.

Then croped center (224x224) and converted into tensors.

Normalization helps training converge faster.

In [29]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


In [30]:
dataset = ImageFolder(f"{path}/PetImages" , transform=transform)

In [31]:
print(dataset.classes)

['Cat', 'Dog']


In [32]:
print(dataset)

Dataset ImageFolder
    Number of datapoints: 24998
    Root location: /kaggle/input/dog-and-cat-classification-dataset/PetImages
    StandardTransform
Transform: Compose(
               Resize(size=224, interpolation=bilinear, max_size=None, antialias=True)
               CenterCrop(size=(224, 224))
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )


In [33]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)


In [34]:
print(len(train_dataset) , len(test_dataset))

19998 5000


In [35]:
train_loaded = DataLoader(dataset=train_dataset , batch_size=128 , shuffle=True ,  num_workers=0 , pin_memory=True)

test_loaded = DataLoader(dataset=test_dataset , batch_size=128 , shuffle=True ,  num_workers=0 , pin_memory=True)

In [36]:
images, labels = next(iter(train_loaded))
print(images.shape , labels.shape )

torch.Size([128, 3, 224, 224]) torch.Size([128])


## CNN Architecture

The network consists of:

Conv2D

↓

ReLU

↓

MaxPool

↓

Conv2D

↓

ReLU

↓

MaxPool

↓

Flatten

↓

Fully Connected

↓

Output

In [37]:
class MyNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.feature = nn.Sequential(
        nn.Conv2d(in_channels=3 , out_channels=32 , kernel_size=3 , padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),

        nn.Conv2d(in_channels=32 , out_channels=64 , kernel_size=3 , padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),

        nn.Conv2d(in_channels=64 , out_channels=128 , kernel_size=3 , padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),

        nn.Conv2d(in_channels=128 , out_channels=256 , kernel_size=3 , padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2),
        nn.AdaptiveAvgPool2d(1)
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),

        nn.Linear(256,128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.2),

        nn.Linear(128, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),

        nn.Linear(32, 1)

    )

  def check(self , x):
    x = self.feature(x).shape
    return x

  def forward(self , x):
    x = self.feature(x)
    x = self.classifier(x)
    return x


In [38]:
model = MyNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()
print(next(model.parameters()).device)

cuda:0


Training Loop


In [39]:
for epoch in range(25):
  total_epoch_loss = 0

  for input , target in train_loaded:
    optimizer.zero_grad()
    input = input.to(device)
    target = target.float().unsqueeze(1).to(device)

    y_pred = model(input)
    loss = loss_fn(y_pred , target)
    loss.backward()
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()
  avg_loss = total_epoch_loss/len(train_loaded)
  print(f"Epoch: {epoch}, Loss: {avg_loss}")


Epoch: 0, Loss: 0.5694769941697455
Epoch: 1, Loss: 0.493992602939059
Epoch: 2, Loss: 0.4489443080060801
Epoch: 3, Loss: 0.4089839733709955
Epoch: 4, Loss: 0.3692382256126708
Epoch: 5, Loss: 0.3317125716786476
Epoch: 6, Loss: 0.30179171586871906
Epoch: 7, Loss: 0.26882486804655403
Epoch: 8, Loss: 0.25213143922341097
Epoch: 9, Loss: 0.226904372405854
Epoch: 10, Loss: 0.2137609949916791
Epoch: 11, Loss: 0.19896658581153603
Epoch: 12, Loss: 0.18226033809838021
Epoch: 13, Loss: 0.1675114387254806
Epoch: 14, Loss: 0.15901315644098696
Epoch: 15, Loss: 0.13799248082907337
Epoch: 16, Loss: 0.13531252039465935
Epoch: 17, Loss: 0.12815566778562631
Epoch: 18, Loss: 0.12546383625098095
Epoch: 19, Loss: 0.10804994330759261
Epoch: 20, Loss: 0.09530488319789908
Epoch: 21, Loss: 0.09850676495368314
Epoch: 22, Loss: 0.0799395050734851
Epoch: 23, Loss: 0.07926248482600519
Epoch: 24, Loss: 0.0832798254148216


Eval

In [41]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
  for input , label in test_loaded:
    input , label = input.to(device) , label.float().to(device)

    logit = model(input)
    pred = torch.sigmoid(logit)
    y_pred = (pred > 0.5).float()

    correct += (y_pred == label.unsqueeze(1)).sum().item()
    total += label.size(0)

  accuracy = (correct / total)*100

print(f"Accuracy of model is {accuracy:02f}%")


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Accuracy of model is 91.360000%
